In [1]:
#!pip install --user antspyx

In [2]:
import sys
import os
import os.path as op
import glob
import nibabel as nib

# Make utils.py (lab helpers) importable
notebook_dir = os.path.abspath("")
sys.path.append(os.path.abspath(os.path.join(notebook_dir, '..')))
sys.path.append('.')
from utils import loadFSL, FSLeyesServer

# Load FSL in Neurodesk
import module as lmod
await lmod.purge(force=True)
await lmod.load('fsl/6.0.7.18')
loadFSL()

from fsl.wrappers import fast, flirt

In [3]:
from nilearn import plotting
fig_dir = op.abspath("outputs/figures")
os.makedirs(fig_dir, exist_ok=True)

In [4]:
%gui wx

In [5]:
################
# Start FSLeyes
################
fsleyesDisplay = FSLeyesServer()
fsleyesDisplay.show()


(ipykernel_launcher.py:1977): dbind-WARNING **: 12:19:20.892: Couldn't connect to accessibility bus: Failed to connect to socket /run/user/203553799/at-spi/bus_100: No such file or directory

(ipykernel_launcher.py:1977): Gtk-CRITICAL **: 12:19:21.201: gtk_window_resize: assertion 'width > 0' failed

(ipykernel_launcher.py:1977): Gtk-CRITICAL **: 12:19:21.212: gtk_window_resize: assertion 'width > 0' failed


In [6]:
t1_path = op.abspath("miniproject_2026_final/T1/T1w.nii.gz")
print(t1_path, op.isfile(t1_path)) 

/neurodesktop-storage/miniproject_2026_final/T1/T1w.nii.gz True


In [7]:
# Visual check of the raw T1w
fsleyesDisplay.resetOverlays()
fsleyesDisplay.load(t1_path)

## 1.1 Skull stripping

In [8]:
def get_skull_stripped_anatomical(anatomical_path, output_dir, robust=True):
    # -R: robust center estimation, avoids leaving neck tissue
    os.makedirs(output_dir, exist_ok=True)
    betted_brain_path = op.join(output_dir, 'T1w')
    os.system('bet {} {} -m {}'.format(anatomical_path, betted_brain_path, '-R' if robust else ''))
    print("Done with BET.")

# paths
anatomical_path = op.abspath("miniproject_2026_final/T1/T1w.nii.gz") #path to the T1w image
output_dir = op.abspath("outputs/anat") #where the results are saved

get_skull_stripped_anatomical(anatomical_path, output_dir)

resulting_mask_path = op.join(output_dir, 'T1w_mask')

Done with BET.


In [9]:
fsleyesDisplay.load(resulting_mask_path)

In [10]:
def apply_python_mask_approach(img_path, mask_path, masked_img_path):

    # Load both the T1 and the mask from disk
    img = nib.load(img_path)
    mask = nib.load(mask_path)
    
    # Load the data from both above images as numpy arrays
    img_data = img.get_fdata()
    mask_data = mask.get_fdata()
    
    # In all positions OUTSIDE the mask (equal to 0), throw away the image
    img_data[mask_data == 0] = 0
    
    # Save the image to disk, by creating a new Nifti image and then writing it out
    img_out = nib.Nifti1Image(img_data,img.affine, img.header)
    nib.save(img_out, masked_img_path)

betted_brain_path = op.join(output_dir, 'T1w_masked.nii.gz') 
resulting_mask_path = op.join(output_dir, 'T1w_mask.nii.gz') # The mask to use

apply_python_mask_approach(anatomical_path, resulting_mask_path, betted_brain_path)

In [ ]:
fsleyesDisplay.resetOverlays()
fsleyesDisplay.load(betted_brain_path)

In [ ]:
# 1a: original vs skull-stripped, with the mask contour on the original
plotting.plot_anat(anatomical_path, title='Original T1w',
                   output_file=op.join(fig_dir, 'T1w_original.png'))
d = plotting.plot_anat(anatomical_path, title='BET mask contour')
d.add_contours(resulting_mask_path, levels=[0.5], colors='r')
d.savefig(op.join(fig_dir, 'T1w_bet_contour.png'))
plotting.plot_anat(betted_brain_path, title='Skull-stripped T1w',
                   output_file=op.join(fig_dir, 'T1w_skullstripped.png'))

## 1.2 Tissue segmentation

In [ ]:
# FAST is applied to the skull-stripped brain: skull and scalp would otherwise be classified as tissue.
fast_target = betted_brain_path   

[os.remove(f) for f in glob.glob(op.join(output_dir, '*fast*'))] # Just to clean the directory in between runs of the cell
segmentation_path = op.join(output_dir, 'T1w_fast')
# n_classes=3 -> CSF, grey matter, white matter (pve_0, pve_1, pve_2)
fast(imgs=[fast_target], out=segmentation_path, n_classes=3)

In [ ]:
fsleyesDisplay.resetOverlays()
fsleyesDisplay.load(betted_brain_path)
fsleyesDisplay.load(glob.glob(op.join(output_dir, '*pve_0*'))[0])
fsleyesDisplay.load(glob.glob(op.join(output_dir, '*pve_1*'))[0])
fsleyesDisplay.load(glob.glob(op.join(output_dir, '*pve_2*'))[0])
fsleyesDisplay.displayCtx.getOpts(fsleyesDisplay.overlayList[1]).cmap = 'Red'
fsleyesDisplay.displayCtx.getOpts(fsleyesDisplay.overlayList[2]).cmap = 'Green'
fsleyesDisplay.displayCtx.getOpts(fsleyesDisplay.overlayList[3]).cmap = 'Blue'

In [ ]:
# 1b: hard segmentation (1=CSF, 2=GM, 3=WM) on the brain
plotting.plot_roi(op.join(output_dir, 'T1w_fast_seg.nii.gz'),
                  bg_img=betted_brain_path, title='FAST segmentation',
                  display_mode='z', cut_coords=6, cmap='Set1',
                  output_file=op.join(fig_dir, 'T1w_segmentation.png'))

## 1.3 Registration

In [ ]:
#from fsl.wrappers import flirt

subject_anatomical = betted_brain_path
mni_template = op.expandvars(op.join('$FSLDIR', 'data', 'standard', 'MNI152_T1_1mm_brain'))

  # Source = subject T1w (moved), reference = MNI152 1mm brain template (fixed).
  # We map the subject to the template so results can be compared in standard space.

source = subject_anatomical 
reference = mni_template
result = op.join(output_dir, 'T1w_mni')
flirt(source, reference, out=result)

Flirt vizualisation : 

In [ ]:
fsleyesDisplay.resetOverlays()
fsleyesDisplay.load(reference) 
fsleyesDisplay.load(result)

Choosing a cost:

In [ ]:
# Same modality (T1w -> T1w template): within-modality intensity-based costs (leastsq, normcorr)
costs = [ 'leastsq', 'normcorr' ] 

for c in costs:
    flirt(source, reference, out=result + '_' + c, cost=c)

In [ ]:
# Before/after: compare the subject T1w and the MNI template overlay
for c in costs:
    fsleyesDisplay.load(result + '_' + c)

In [ ]:
#from nilearn import plotting
mni_path = mni_template + '.nii.gz'
cuts = (0, -20, 10)   # x, y, z in mm, identical for all panels

images = {
    'Before registration': betted_brain_path,
    'After - corratio (default)': op.join(output_dir, 'T1w_mni.nii.gz'),
    'After - leastsq': op.join(output_dir, 'T1w_mni_leastsq.nii.gz'),
    'After - normcorr': op.join(output_dir, 'T1w_mni_normcorr.nii.gz'),
}

for title, path in images.items():
    d = plotting.plot_anat(path, title=title, cut_coords=cuts)
    d.add_edges(mni_path)

In [ ]:
final_cost = 'normcorr'   # the one we keep
mat_path = op.join(output_dir, 'T1w_to_MNI.mat')
flirt(source, reference, out=op.join(output_dir, 'T1w_mni_final'),
      cost=final_cost, omat=mat_path)

In [ ]:
for name, interp in [('seg', 'nearestneighbour'),
                     ('pve_0', 'trilinear'), ('pve_1', 'trilinear'), ('pve_2', 'trilinear')]:
    flirt(op.join(output_dir, f'T1w_fast_{name}.nii.gz'), reference,
          out=op.join(output_dir, f'T1w_fast_{name}_mni'),
          applyxfm=True, init=mat_path, interp=interp)

In [ ]:
# List outputs to check all files were created
!ls -lh {output_dir}

## 2. Functional preprocessing

### a) Motion correction

The first step to do before correcting motion is to inspect our data on fsleye to see if there is any abnormal volume that we could remove

In [ ]:
import os, os.path as op
import nibabel as nib
import matplotlib.pyplot as plt

project_root = op.abspath("")
data_root     = op.join(project_root, "miniproject_2026_final")
fMRI_raw     = op.join(data_root, "fMRI")
fMRI_lang_raw = op.join(fMRI_raw, "tfMRI_LANGUAGE_LR")
func_file    = op.join(fMRI_lang_raw, "tfMRI_LANGUAGE_LR.nii.gz")   

4D visualisation : 

In [ ]:
fsleyesDisplay.resetOverlays()
fsleyesDisplay.load(func_file)

We can also quantify the abnormal volumes by computing and ploting the mean intensity per volume : 

In [ ]:
img  = nib.load(func_file)
data = img.get_fdata()
print(img.shape)                      # (x, y, z, n_volumes)

mean_per_vol = data.mean(axis=(0, 1, 2))

plt.figure(figsize=(10, 3))
plt.plot(mean_per_vol)
plt.xlabel('Time (volume)')
plt.ylabel('Mean voxel intensity')
plt.show()

If we look at the plot we can clearly see that the first ~ 3 volumes have a high contrast that quickly decayed to some baseline. This is caused by the scanner's field settling. Around volumes 210-250, the mean intensity shows a peak followed by a drop, we will compare this with the FD plot to see whether it coincides with head motion.
In the next step, we remove the first 10 volumes that are impacted by field stabilisation. 


In [ ]:
from fsl.wrappers import fslroi, mcflirt

n_discard = 10        # volumes we remove
n_vols    = img.shape[3]

out_func = op.join(project_root, "outputs", "func")
os.makedirs(out_func, exist_ok=True)

trimmed = op.join(out_func, "tfMRI_LANGUAGE_LR_trim")          
moco    = op.join(out_func, "tfMRI_LANGUAGE_LR_trim_moco")


fslroi(func_file, trimmed, n_discard, n_vols - n_discard) #trimmedcorrespond to the data without the first 10 volumes

Finally, we can run MCFLIRT to correct motion

In [ ]:
mcflirt(infile=trimmed, o=moco, plots=True, mats=True, dof=6, report=True)

Framewise displacement :

Framewise displacement was computed from the six rigid-body motion parameters estimated by MCFLIRT on the trimmed data.

In [ ]:
import os, os.path as op
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 1. Charger les paramètres de mouvement écrits par MCFLIRT (fichier .par)

motion_params_path = op.join(project_root, "outputs", "func", "tfMRI_LANGUAGE_LR_trim_moco.par") # 6 colonnes : 3 rotations (rad), 3 translations (mm)
print(op.exists(motion_params_path))   

mot_params = pd.read_csv(motion_params_path, sep=r"\s+", header=None,
                         names=['Rotation x', 'Rotation y', 'Rotation z',
                                'Translation x', 'Translation y', 'Translation z'])

# 2. Calcul du FD (Power et al.)
def compute_FD_power(mot_params):
    framewise_diff = mot_params.diff().iloc[1:]          # différence entre volumes consécutifs

    rot_params = framewise_diff[['Rotation x', 'Rotation y', 'Rotation z']]
    converted_rots = rot_params * 50                     # radians -> mm (arc sur une sphère de 50 mm)

    trans_params = framewise_diff[['Translation x', 'Translation y', 'Translation z']]

    return converted_rots.abs().sum(axis=1) + trans_params.abs().sum(axis=1)

fd = compute_FD_power(mot_params).to_numpy()

# 3. Seuil d'outliers : Q3 + 1.5 * IQR
q1, q3 = np.quantile(fd, [0.25, 0.75])
threshold = q3 + 1.5 * (q3 - q1)


In [ ]:
# 4. Graphique
plt.figure(figsize=(10, 3))
plt.plot(range(1, fd.size + 1), fd)
plt.hlines(threshold, 0, fd.size + 1, colors='red', linestyles='dashed', label='FD threshold')
plt.xlabel('Volume')
plt.ylabel('FD displacement (mm)')
plt.legend()

# 5. Sauvegarde de la figure
fig_dir = op.join(project_root, "outputs", "figures")
os.makedirs(fig_dir, exist_ok=True)
plt.savefig(op.join(fig_dir, "fd_tfMRI_LANGUAGE_LR.png"), dpi=150, bbox_inches="tight")
plt.show()

# 6. Volumes au-dessus du seuil
bad_vols = (np.where(fd > threshold)[0] + 1).tolist()
print(f"FD Treshold : {threshold:.3f} mm")
print(f"Indices of frames above threshold: {bad_vols}")

**Comments :** 

The framewise displacement (FD) remains low throughout the run: most volumes show an FD below the threshold (0.222 mm), with a maximum of 0.34 mm at volume 51. Using an outlier criterion (Q3 + 1.5 × IQR = 0.222 mm), only the following five volumes : **51, 69, 83, 128 and 230** exceed the threshold. These are isolated single-volume spikes rather than sustained periods of motion, which suggests that the participant moved very little. Also when we displayed the volumes on fsleye we could not see any significant motion for those volumes (by eye) compared to the previous/next one.

Since this threshold is data-driven, it flags relative outliers even when absolute motion is small. The signal changes observed in the mean intensity around volumes 210-250 of the raw series do not coincide with a clear FD peak, so they are probably not mainly driven by head motion, although FD only captures volume-to-volume displacement.

Motion correction realigns the volumes but cannot remove the residual, non-linear effects of motion on the signal (e.g. through changes in the magnetic field). These can be reduced later by including the six motion parameters, and if needed one regressor per flagged volume, as nuisance regressors in the GLM.

Motion between volume 51 and 50 :

In [ ]:
from nilearn import plotting, image
import matplotlib.pyplot as plt

vol = 51
trim_func_base = op.join(project_root, "outputs", "func", "tfMRI_LANGUAGE_LR_trim")
moco_func_base = op.join(project_root, "outputs", "func", "tfMRI_LANGUAGE_LR_trim_moco")
trim_vol = image.index_img(trim_func_base + ".nii.gz", vol)  
moco_vol = image.index_img(moco_func_base + ".nii.gz", vol)      
ref_vol  = image.index_img(moco_func_base + ".nii.gz", vol - 1)  

diff = image.math_img("a - b", a=trim_vol, b=image.index_img(trim_func_base + ".nii.gz", vol - 1))
plotting.plot_stat_map(diff, title=f"Volume {vol} minus {vol-1} (trimmed)")

The figure shows the voxel-wise difference between volume 51 and volume 50 (before motion correction)

### b) Gaussian smoothing

FWHM choice : 

In [ ]:
img = nib.load(moco_func_base + ".nii.gz")
print("Voxel size (mm):", img.header.get_zooms()[:3])
print("TR (s):", img.header.get_zooms()[3])
print("Shape:", img.shape)

# ! explain our choice 

In [ ]:
import subprocess
#from nilearn import plotting, image

fwhm_mm = 6
sigma = fwhm_mm / 2.3548
smoothed_func_base = f"{moco_func_base}_smoothed-{fwhm_mm}mm"

subprocess.run(["fslmaths", moco_func_base, "-s", str(sigma), smoothed_func_base], check=True)

Figure : 

In [ ]:
# 4. Before / after for one volume (same slices, same intensity scale)
vol = 100                                               # any volume of your choice
before = image.index_img(moco_func_base + ".nii.gz", vol)
after  = image.index_img(smoothed_func_base + ".nii.gz", vol)
vmax   = float(before.get_fdata().max())

fig, axes = plt.subplots(2, 1, figsize=(10, 7))
for ax, img_, title in zip(axes, [before, after],
                           [f"Volume {vol} before smoothing", f"Volume {vol} after smoothing ({fwhm_mm} mm FWHM)"]):
    plotting.plot_epi(img_, axes=ax, title=title, cmap="gray", vmax=vmax,
                      display_mode="ortho", cut_coords=(0, 0, 0), colorbar=False)

fig_dir = op.join(project_root, "outputs", "figures")
os.makedirs(fig_dir, exist_ok=True)
fig.savefig(op.join(fig_dir, f"smoothing_vol{vol}.png"), dpi=150, bbox_inches="tight")
plt.show()

### c) Validate the alignment between anatomical and functional data

In [ ]:
import os, os.path as op
import numpy as np
import nibabel as nib
from nilearn import plotting, image

# 1. Files
epi_path = op.join(project_root, "outputs", "func", "tfMRI_LANGUAGE_LR_trim_moco.nii.gz")   # motion-corrected fMRI
print(os.listdir(op.join(project_root, "outputs", "anat")))     # find your T1 registered to MNI
t1_mni_path = op.join(project_root, "outputs", "anat", "YOUR_T1_IN_MNI.nii.gz")              # <- fill in

# 2. Are they in the same space?
epi_img, t1_img = nib.load(epi_path), nib.load(t1_mni_path)
print("fMRI:", epi_img.shape[:3], epi_img.header.get_zooms()[:3])
print("T1  :", t1_img.shape[:3], t1_img.header.get_zooms()[:3])
print("Same affine:", np.allclose(epi_img.affine, t1_img.affine, atol=1e-3))


In [ ]:
# 3. Overlay: one fMRI volume (grey) with the T1 edges (red)
vol = 100
epi_vol = image.index_img(epi_img, vol)

display = plotting.plot_epi(epi_vol, display_mode="ortho", cut_coords=(0, -20, 10),
                            cmap="gray", title=f"fMRI volume {vol} with T1 (MNI) edges in red")
display.add_edges(t1_mni_path, color="r")

fig_dir = op.join(project_root, "outputs", "figures")
os.makedirs(fig_dir, exist_ok=True)
display.savefig(op.join(fig_dir, f"overlay_fmri_t1_vol{vol}.png"), dpi=150)
plotting.show()